# Feature Track 4c: RAG as a Tool

Instead of always retrieving chunks before answering, the LLM can be given retrieval as an **optional tool**. It then decides when to call it, answering from its own knowledge for general questions, and retrieving chunks for domain-specific ones.

This notebook builds the retrieval tool and runs the call-execute-respond loop manually (as in 4a). Notebook 4d wraps this in a `ToolAgent`.

---

## Setup

**Prerequisites:** `conversational_toolkit` and `backend` installed in editable mode. Set `OPENAI_API_KEY`.

The vector store is built or reused from disk (`VS_PATH`). The retriever wraps the vector store with the embedding model.

In [ ]:
import json
from conversational_toolkit.embeddings.sentence_transformer import (
    SentenceTransformerEmbeddings,
)
from conversational_toolkit.embeddings.openai import OpenAIEmbeddings
from conversational_toolkit.llms.base import LLMMessage, Roles, MessageContent
from conversational_toolkit.llms.openai import OpenAILLM
from conversational_toolkit.vectorstores.chromadb import ChromaDBVectorStore

from conversational_toolkit.retriever.vectorstore_retriever import VectorStoreRetriever

from sme_kt_zh_collaboration_rag.feature0_baseline_rag import (
    load_chunks,
    build_llm,
    build_vector_store,
    VS_PATH,
    EMBEDDING_MODEL,
)

from sme_kt_zh_collaboration_rag.feature4_tool_agents import RetrieveRelevantChunks


BACKEND = "openai"  # "ollama" or "openai"
FORCE_REBUILD = False  # set True to re-embed from scratch

if "sentence-transformers" in EMBEDDING_MODEL:
    embedding_model = SentenceTransformerEmbeddings(model_name=EMBEDDING_MODEL)
else:
    embedding_model = OpenAIEmbeddings(model_name=EMBEDDING_MODEL)

if FORCE_REBUILD or not VS_PATH.exists():
    chunks = load_chunks(max_files=5)
    chunks = [c for c in chunks if c.mime_type.startswith("text")]
    db_chroma = await build_vector_store(
        chunks, embedding_model, db_path=VS_PATH, reset=FORCE_REBUILD
    )
else:
    # Vector store already exists -> open it without re-embedding
    db_chroma = ChromaDBVectorStore(db_path=str(VS_PATH))
    print(
        f"Reusing existing vector store at {VS_PATH} ({db_chroma.collection.count()} chunks)"
    )

vector_store = VectorStoreRetriever(embedding_model, db_chroma, top_k=5)

if not BACKEND:
    raise ValueError('Set BACKEND to "ollama" or "openai" before running.')

llm = build_llm(backend=BACKEND)

---

## Define the Retrieval Tool

The tool takes a query string, calls the vector store retriever, and returns the matching chunks formatted as text. The LLM uses the tool description to decide when to call it.

In [ ]:
retriever_tool = RetrieveRelevantChunks(
    name="retrieve_relevant_chunks",
    description="Retrieves the most relevant chunks based on a query.",
    # What parameters it expects
    parameters={
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "The query to retrieve relevant chunks for.",
            },
        },
        "required": ["query"],
        "additionalProperties": False,
    },
    retriever=vector_store,
)

In [ ]:
# Test if it works
result = await retriever_tool.call(
    {"query": "Which pallets in our portfolio have a third-party verified EPD?"}
)

print("\nResults: ", result["result"][0])

---

## Provide the Tool to the LLM

The system prompt instructs the LLM to use retrieved chunks as its only source of truth for domain questions. The prompt template wraps user queries in a consistent format.

In [ ]:
system_prompt = """You are a helpful assistant that answers questions.

You have access to the following tool:
- retrieve_relevant_chunks: Retrieves the most relevant chunks based on a query.

Only use the tool if it's relevant, else answer based on your own knowledge. Always try to use the tool if you think it can help you answer the question better.

If you use the tool, follow these guidelines:
- Use the chunks as your only source of truth. Do not rely on outside knowledge.
- Use all relevant chunks when forming your answer. Do not ignore any provided information.
- If the answer cannot be found in the chunks, clearly say that you do not know.
- Keep your answer concise and focused, without unnecessary details.
- Cite your sources from the provided chunks."""

prompt_message = LLMMessage(
    content=[MessageContent(type="text", text=system_prompt)], role=Roles.SYSTEM
)

prompt_template = """# User question:\n{question}\n\nYour answer:\n\n"""

llm = OpenAILLM(tools=[retriever_tool], tool_choice="auto")

## Test

Two queries show the LLM's tool-use decision: a general knowledge question (no tool needed) and a domain-specific question (tool called). Notice that the LLM may rewrite the query before calling the tool.

### General Question

In [ ]:
query = "What is Einstein's theory of relativity? Answer concisely in 2-3 sentences."
user_message = LLMMessage(
    role=Roles.USER,
    content=[MessageContent(type="text", text=prompt_template.format(question=query))],
)

response = await llm.generate(conversation=[prompt_message, user_message])

# Here the LLM should not call the tool, as it can answer based on its own knowledge
print("\nResponse: ", response.content[0].text)
print("\nTool calls: ", response.tool_calls)
print("\nTool call ID: ", response.tool_call_id)

### Normal Question

Note: When the LLM decides to call a tool, it does not generate an answer in that same response. The response contains only the tool call request, and content is empty. The answer comes in a second LLM call after the tool result is fed back.

In [ ]:
# Let's ask a question about PrimePack AG
query = "Which pallets in our portfolio have a third-party verified EPD?"
user_message = LLMMessage(
    role=Roles.USER,
    content=[MessageContent(type="text", text=prompt_template.format(question=query))],
)

response = await llm.generate(conversation=[prompt_message, user_message])

# Here the LLM indeed calls the tool. Note that the query is not the same, the LLM rewrote it.
print(f"\n{response}")
print("\nResponse: ", response.content[0].text)
print("\nTool calls: ", response.tool_calls)
print("\nTool call ID: ", response.tool_call_id)

In [ ]:
# Look at which chunks where retrieved

results = {}

assert response.tool_calls is not None
assert llm.tools is not None

for tool_call in response.tool_calls:
    tool_name = tool_call.function.name
    tool_args = tool_call.function.arguments

    tool = next((t for t in llm.tools if t.name == tool_name), None)

    if tool is not None:
        tools_args_json = json.loads(tool_args)
        tool_result = await tool.call(tools_args_json)

        results[tool_name] = tool_result

print("Retrieved chunks: \n", results["retrieve_relevant_chunks"]["result"][0])

In [ ]:
# Final response

tools_answers = []

assert response.tool_calls is not None

for tool_call in response.tool_calls:
    tool_name = tool_call.function.name
    result = results[tool_name]
    call_id = tool_call.id

    tool_answer = LLMMessage(
        role=Roles.TOOL,
        name=tool_name,
        content=[MessageContent(text=json.dumps(result), type="text")],
        tool_call_id=call_id,
    )
    tools_answers.append(tool_answer)

conversation = [user_message, response, *tools_answers]

final_response = await llm.generate(conversation)

print("\nFinal response: \n", final_response.content[0].text)

--------------------------